# Track 1 · 11 REVE + LoRA on Alljoined-1.6M — the sealed-montage model

**One job.** Fine-tune REVE (frozen, with LoRA adapters) on Alljoined-1.6M, the corpus the
sealed cohort is drawn from (32-electrode Emotiv, 256 Hz), and judge it on whole people it
never saw. Needs `track1_12_alljoined_prepare` to have run. Self-contained: upload this one file.

Design agreed with the user, 2026-11-13 (PLAN.md section 12, "Track 1 Alljoined design").

## The model

- REVE-base frozen; LoRA beside the 44 attention projections and the 44 feed-forward matrices
  (run E/Y recipe, no new sweep).
- **Order-free head:** LayerNorm, one shared Linear 512 -> 64 applied to every electrode's
  token, GELU, dropout, **mean over the electrodes**, one Linear 64 -> 1536. No weight belongs to
  a particular electrode (`head_kind = 'funnelpool'`, in `solvers/track1_image_reve_lora_perchan.py`).
- Loss: InfoNCE against the DINOv2-giant image embeddings, temperature 0.07 (as in track1_09).

## Three views of every training window (mean loss over the three passes)

1. the window as it is;
2. a copy with Gaussian noise (`NOISE_FRAC` of each channel's own sd);
3. a copy with the **channels shuffled** (signals moved between electrodes, a fresh random
   order per window, time axis untouched; each electrode's position stays where it was).
   This is regularisation, not a way to learn positions: positions only tell REVE where a
   signal sits, and a shuffle puts signals on the wrong places.

`AUGMENT = False` trains on the originals only, as a control. Three views cost about three
times the compute per epoch; epoch 1 prints its time and a projected total.

## Judging

Whole people are held out (`N_HOLDOUT_PEOPLE`): never fitted, never validated, scored at the end
as subject-aggregated top-5 on the 200 fixed Alljoined test images. The local gallery is 200
images (chance top-5 2.5%); the sealed gallery is much larger (our inference: about 8,000), so
compare methods here by rank and top-5, not against the sealed board. The best-validation epoch's weights are
kept (`KEEP = 'best'`, saved to Drive each time it improves), and training stops early when validation rank
has not gained for `PATIENCE` epochs or the validation loss climbs away from its minimum
(`DIVERGE_LOSS`); `EPOCHS` is only the upper bound. `KEEP = 'last'` keeps the final epoch instead.


## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'
DATA = 'alljoined'               # Track1/<DATA>/ written by track1_12_alljoined_prepare

N_HOLDOUT_PEOPLE = 5             # whole people never fitted or validated; scored at the end
HOLDOUT_PEOPLE = []              # or name them (subject ids as in the file names, e.g. ['03', '11'])

EPOCHS = 20
HEAD_ONLY_EPOCHS = 1             # adapters stay frozen for this many epochs
BATCH = 256
LR_ADAPTER, LR_HEAD = 1e-4, 1e-3
LORA_RANK, LORA_ALPHA = 8, 8             # attention (to_qkv, to_out)
LORA_RANK_FFN, LORA_ALPHA_FFN = 8, 8     # feed-forward (net.1, net.3)
HEAD = 'funnelpool'
SHRINK = 64                      # the funnel width: 512 -> 64 per electrode
DROPOUT = 0.4
TEMPERATURE = 0.07
USE_AMP = True                   # mixed precision on GPU

AUGMENT = True                   # original + noise copy + channel-shuffled copy (False: control)
NOISE_FRAC = 0.1                 # noise sd as a fraction of each channel's sd
KEEP = 'best'                    # 'best' validation-rank epoch (also saved to Drive as it happens), or 'last'
PATIENCE = 3                     # early stop: epochs in a row without a validation-rank gain
MIN_DELTA = 0.0005               # a gain smaller than this does not count
DIVERGE_LOSS = 0.1               # early stop: val loss above its minimum by this, 2 epochs in a row

MAKE_ZIP = True
VAL_IMAGE_FRAC = 0.2
SEED = 33

RUN_TAG = (f'reve_alljoined_r{LORA_RANK}_a{LORA_ALPHA}_s{SHRINK}_d{DROPOUT}_b{BATCH}'
           + (f'_ffn{LORA_RANK_FFN}x{LORA_ALPHA_FFN}' if LORA_RANK_FFN else '')
           + (f'_aug{NOISE_FRAC:g}' if AUGMENT else '_noaug') + f'_{KEEP}')
RUN_TRAINING = True
print(f'NOTEBOOK VERSION: track1_11 v1   settings {RUN_TAG}   epochs {EPOCHS}   '
      f'chance loss will be {__import__("math").log(BATCH):.4f}')


In [ ]:
%pip install -q 'braindecode==1.8.1' huggingface_hub safetensors

import importlib
for mod in ('braindecode', 'torch'):
    print(f'{mod:12} {getattr(importlib.import_module(mod), "__version__", "ok")}')

In [ ]:
import json, os, shutil, sys, time, types, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
BASE = ROOT / 'Track1'
AJ = BASE / DATA
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', '/content/hf')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = USE_AMP and DEVICE == 'cuda'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def l2(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)


def _truth(y_ids):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    return gal, np.array([pos[y] for y in y_ids])


def rank_pct(pred, y_ids, emb):
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    return float((sims < sims[np.arange(len(sims)), truth][:, None]).mean())


def top5(pred, y_ids, emb):
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    k = min(5, len(gal))
    tk = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    return 100.0 * np.mean([t in r for t, r in zip(truth, tk)]), 100.0 * k / len(gal)


def rank_null(pred, y_ids, n=200):
    """Validation rank under permuted labels, for these exact predictions."""
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    rows = np.arange(len(sims))
    return np.array([float((sims < sims[rows, p][:, None]).mean())
                     for p in (rng.permutation(truth) for _ in range(n))])


print(f'device {DEVICE}, mixed precision {AMP}')

## Step 1 — the Alljoined windows and targets

In [ ]:
t = np.load(AJ / 'image_embeddings.npz', allow_pickle=True)
emb = l2(t['emb'].astype('float32'))
DIM = emb.shape[1]
key = {k: i for i, k in enumerate(t['keys'].tolist())}

train_files = sorted((AJ / 'windows').glob('*_train.npz'))
test_files = sorted((AJ / 'windows').glob('*_test.npz'))
if not train_files or not test_files:
    raise FileNotFoundError(f'no windows in {AJ / "windows"}; run track1_12_alljoined_prepare first')

# Pass 1: labels, people and which windows each file contributes (small).
plan, ys, subs, CH = [], [], [], None
for f in train_files:
    z = np.load(f, allow_pickle=True)
    idx = np.array([key.get(k_, -1) for k_ in z['keys'].tolist()])
    keep = idx >= 0
    if CH is None:
        CH = list(z['channels'])
    elif list(z['channels']) != CH:
        raise RuntimeError(f'{f.name}: channel order differs from the first training file')
    plan.append((f, keep))
    ys.append(idx[keep])
    subs.append(np.full(int(keep.sum()), str(z['subject'])))
n_train = sum(int(k.sum()) for _, k in plan)

# Pass 2: the windows themselves, written once into one array on the machine's local disk
# (not Drive) and read per batch; a rerun in the same session reuses it.
local = Path('/content') if Path('/content').exists() else AJ
xfile, sigfile = local / 'track1_aj_train_X.npy', local / 'track1_aj_train_X.json'
signature = json.dumps([[str(f), f.stat().st_size, int(k.sum())] for f, k in plan])
if xfile.exists() and sigfile.exists() and sigfile.read_text() == signature:
    Xtr = np.load(xfile, mmap_mode='r')
    print(f'reusing {xfile} ({xfile.stat().st_size / 1e9:.1f} GB)')
else:
    Xtr, i, t0 = None, 0, time.time()
    for f, keep in plan:
        x = np.load(f)['X'][keep]
        if Xtr is None:
            Xtr = np.lib.format.open_memmap(xfile, mode='w+', dtype='float16',
                                            shape=(n_train,) + x.shape[1:])
        Xtr[i:i + len(x)] = x
        i += len(x)
        del x
    Xtr.flush()
    sigfile.write_text(signature)
    Xtr = np.load(xfile, mmap_mode='r')
    print(f'{n_train:,} training windows from {len(plan)} files written to {xfile} '
          f'in {time.time() - t0:.0f}s')
Wtr = [Xtr, np.concatenate(ys), np.concatenate(subs)]

parts = [[], [], []]
for f in test_files:
    z = np.load(f, allow_pickle=True)
    idx = np.array([key.get(k_, -1) for k_ in z['keys'].tolist()])
    keep = idx >= 0
    parts[0].append(z['X'][keep]); parts[1].append(idx[keep])
    parts[2].append(np.full(int(keep.sum()), str(z['subject'])))
    if list(z['channels']) != CH:
        raise RuntimeError(f'{f.name}: test channel order differs from training')
Wte = [np.concatenate(a) for a in parts]
IN_SFREQ = float(json.loads((AJ / 'manifest.json').read_text())['sfreq'])

people = sorted(set(Wtr[2].tolist()))
if not HOLDOUT_PEOPLE:
    HOLDOUT_PEOPLE = sorted(rng.choice(people, N_HOLDOUT_PEOPLE, replace=False).tolist())
HOLDOUT_PEOPLE = [str(p_) for p_ in HOLDOUT_PEOPLE]
assert set(HOLDOUT_PEOPLE) <= set(people), f'unknown people {set(HOLDOUT_PEOPLE) - set(people)}'
SEEN = ~np.isin(Wtr[2], HOLDOUT_PEOPLE)        # rows allowed into fit AND val
seen_imgs = np.unique(Wtr[1][SEEN])
val_imgs = set(rng.choice(seen_imgs, int(len(seen_imgs) * VAL_IMAGE_FRAC), replace=False).tolist())
IV = np.array([y in val_imgs for y in Wtr[1]])
print(f'{len(people)} people; HELD OUT {HOLDOUT_PEOPLE}: {(~SEEN).sum():,} windows never fitted or validated')
print(f'train {Wtr[0].shape}   test {Wte[0].shape}   {len(CH)} channels at {IN_SFREQ:g} Hz')
print(f'{((~IV) & SEEN).sum():,} fit / {(IV & SEEN).sum():,} val windows, '
      f'{len(np.unique(Wte[1]))} test images')


## Step 2 — the model, from the submission file

The submission file is executed here to get the model class, so what is trained is exactly what is submitted.

In [ ]:
# Source of truth: solvers/track1_image_reve_lora_perchan.py in the repo.
SUBMISSION_PY = '"""Track 01 submission: REVE with LoRA adapters and a per-electrode head.\n\nThe encoder is `brain-bzh/reve-base`. Its pretrained weights are frozen; LoRA learns a\nsmall rank-r correction beside each of the 44 attention projections\n(`transformer.layers.*.0.{to_qkv,to_out}`) and, when `ffn_rank` is set, beside the two\nfeed-forward matrices in each block (`transformer.layers.*.1.net.{1,3}`), so no\npretrained weight is overwritten.\n\nThe head gives every electrode its own weights (the default, `head_kind="perelec"`), or\npools the channel tokens with K learned queries (`head_kind="mqpool"`), or applies one shared\nfunnel to every electrode and averages (`head_kind="funnelpool"`, order-free), which keeps a\nfixed K x embed_dim bottleneck at any channel count. Averaging electrodes together, or\napplying one shared map to each, both lost the signal on this task, and each electrode\nneeding its own mapping was the consistent finding. Each electrode\'s 512-d token is\nshrunk to a few numbers by one shared layer, then a final layer with separate weights\nper electrode maps them to the target embedding. The shrink keeps that layer small.\n\nThat makes the head specific to the electrodes it was trained on. Inputs are matched to\nthem by name: extra electrodes are ignored and missing ones are filled with zeros, which\nafter the harness\'s scaling means "no information". Electrodes REVE\'s position bank does\nnot know are left out, at training and at inference alike.\n\nInput is the harness\'s own preprocessing (120 Hz, RobustScaler, baseline, clamp),\nresampled to REVE\'s 200 Hz and standardised per window inside the model, so the\ntraining windows and the scoring windows go through identical code.\n\nThis file is also the model definition the training notebook uses, so the two cannot\ndrift apart.\n"""\n\nimport json\nimport os\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import REVE\n\nREVE_SFREQ = 200.0     # REVE\'s pretraining rate\nPATCH = 200            # REVE\'s patch_size in samples\nCLIP_SD = 15.0\nLORA_TARGETS = ("to_qkv", "to_out")\n# FeedForward.net is Sequential(RMSNorm, Linear, GEGLU, Linear), so its two weight\n# matrices sit at net.1 and net.3 (net.0 is the norm, net.2 the activation).\nFFN_TARGETS = (".net.1", ".net.3")\n\n\ndef resample_fft(x, n_out):\n    """Resample along time via the rfft spectrum. Pure torch, since nothing is\n    installed at evaluation and linear interpolation aliases."""\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass LoRALinear(nn.Module):\n    """Frozen base plus a rank-r update. B starts at zero, so training begins at\n    exactly the pretrained function."""\n\n    def __init__(self, base, r, alpha):\n        super().__init__()\n        self.base = base\n        for p in self.base.parameters():\n            p.requires_grad_(False)\n        self.A = nn.Parameter(torch.zeros(r, base.in_features))\n        self.B = nn.Parameter(torch.zeros(base.out_features, r))\n        nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)\n        self.scale = alpha / r\n\n    def forward(self, x):\n        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale\n\n    @property\n    def weight(self):\n        """Effective weight, W + scale * B A. braindecode >= 2.0 reads `net[1].weight` of the\n        feed-forward GEGLU directly instead of calling the module, which would skip the\n        adapter; with this property the adapter is applied there too. (The input dropout of\n        this adapter does not apply on that path.)"""\n        return self.base.weight + (self.B @ self.A) * self.scale\n\n\nclass PerElectrodeHead(nn.Module):\n    """Shared shrink per electrode, then separate weights per electrode."""\n\n    def __init__(self, n_chans, embed_dim, shrink, n_outputs, dropout):\n        super().__init__()\n        self.norm = nn.LayerNorm(embed_dim)\n        self.shrink = nn.Linear(embed_dim, shrink)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(n_chans * shrink, n_outputs)\n\n    def forward(self, tokens):                    # (B, chan, embed)\n        z = self.drop(F.gelu(self.shrink(self.norm(tokens))))\n        return self.out(z.flatten(1))\n\n\nclass MQPoolHead(nn.Module):\n    """K learned queries, each pooling the channel tokens by a content-weighted\n    average (divide: one mixture per query; conquer: concat; aggregate: linear).\n    The bottleneck is K * embed_dim regardless of channel count, so a 32-channel\n    input flows through the same head as a 63-channel one."""\n\n    def __init__(self, k, embed_dim, n_outputs, dropout):\n        super().__init__()\n        self.k = k\n        self.queries = nn.Parameter(torch.randn(k, embed_dim) / embed_dim ** 0.5)\n        self.norm = nn.LayerNorm(embed_dim)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(k * embed_dim, n_outputs)\n\n    def forward(self, tokens):                    # (B, chan, embed)\n        z = self.norm(tokens)\n        w = torch.einsum(\'ke,bce->bkc\', self.queries, z) / self.queries.shape[-1] ** 0.5\n        w = w.softmax(dim=-1)                     # (B, K, chan), from this window\'s content\n        pooled = torch.einsum(\'bkc,bce->bke\', w, z)\n        return self.out(self.drop(F.gelu(pooled.flatten(1))))\n\n\nclass FunnelPoolHead(nn.Module):\n    """Order-free head: one shared funnel per electrode token (embed -> shrink), a mean\n    over electrodes, then one linear map to the target. No weight belongs to a particular\n    electrode, so shuffling the electrodes cannot change the output."""\n\n    def __init__(self, embed_dim, shrink, n_outputs, dropout):\n        super().__init__()\n        self.norm = nn.LayerNorm(embed_dim)\n        self.shrink = nn.Linear(embed_dim, shrink)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(shrink, n_outputs)\n\n    def forward(self, tokens):                    # (B, chan, embed)\n        z = self.drop(F.gelu(self.shrink(self.norm(tokens))))\n        return self.out(z.mean(dim=1))\n\n\nclass RevePerChannel(nn.Module):\n\n    def __init__(self, channels, n_outputs, in_sfreq, rank=8, alpha=16, shrink=32,\n                 dropout=0.3, encoder_state=None, ffn_rank=0, ffn_alpha=0,\n                 head_kind="perelec", mq_k=2):\n        super().__init__()\n        self.in_sfreq = float(in_sfreq)\n        self.net = REVE(\n            n_outputs=n_outputs, n_chans=len(channels), n_times=PATCH,\n            sfreq=REVE_SFREQ, chs_info=[{"ch_name": c} for c in channels],\n        )\n        self.net.final_layer = nn.Identity()\n        if encoder_state is not None:             # training: pretrained weights first\n            self.net.load_state_dict(encoder_state, strict=False)\n        for p in self.net.parameters():\n            p.requires_grad_(False)\n        names = [\n            n for n, m in self.net.named_modules()\n            if isinstance(m, nn.Linear) and n.split(".")[-1] in LORA_TARGETS\n        ]\n        for name in names:\n            parent = self.net.get_submodule(name.rsplit(".", 1)[0])\n            setattr(parent, name.split(".")[-1],\n                    LoRALinear(self.net.get_submodule(name), rank, alpha))\n        self.n_lora = len(names)\n        self.n_ffn = 0\n        if ffn_rank:                    # the feed-forward matrices, two per block\n            ffn_names = [\n                n for n, m in self.net.named_modules()\n                if isinstance(m, nn.Linear) and n.endswith(FFN_TARGETS)\n                and n.startswith("transformer.layers.")\n            ]\n            for name in ffn_names:\n                parent = self.net.get_submodule(name.rsplit(".", 1)[0])\n                setattr(parent, name.split(".")[-1],\n                        LoRALinear(self.net.get_submodule(name), ffn_rank, ffn_alpha))\n            self.n_ffn = len(ffn_names)\n\n        # The position bank returns only names it recognises and merely logs the\n        # shortfall, which later breaks the forward. Resolve here, identically for\n        # training and inference, and leave unknown electrodes out.\n        bank = self.net._position_bank.mapping\n        folded = {k.casefold(): k for k in bank}\n        keep, resolved = [], []\n        for i, ch in enumerate(channels):\n            name = ch if ch in bank else folded.get(ch.casefold())\n            if name is not None:\n                keep.append(i)\n                resolved.append(name)\n        if not keep:\n            raise RuntimeError(f"no electrode resolved in REVE\'s bank: {channels[:8]}")\n        self.used = resolved\n        self.register_buffer("keep_idx", torch.tensor(keep), persistent=False)\n        self.register_buffer("pos", self.net.get_positions(resolved), persistent=False)\n        self.head = (FunnelPoolHead(self.net.embed_dim, shrink, n_outputs, dropout)\n                     if head_kind == "funnelpool" else\n                     MQPoolHead(mq_k, self.net.embed_dim, n_outputs, dropout)\n                     if head_kind == "mqpool" else\n                     PerElectrodeHead(len(keep), self.net.embed_dim, shrink,\n                                      n_outputs, dropout))\n\n    def adapter_parameters(self):\n        return [p for n, p in self.named_parameters() if n.endswith((".A", ".B"))]\n\n    def forward(self, X):                          # (B, len(channels), T) at in_sfreq\n        X = X.float().index_select(1, self.keep_idx)\n        n_out = max(int(round(X.shape[-1] * REVE_SFREQ / self.in_sfreq)), PATCH)\n        X = resample_fft(X, n_out)\n        X = (X - X.mean(-1, keepdim=True)) / (X.std(-1, keepdim=True) + 1e-8)\n        X = torch.clamp(X, -CLIP_SD, CLIP_SD)\n        pos = self.pos.unsqueeze(0).expand(X.shape[0], -1, -1)\n        feats = self.net(X, pos=pos, return_features=True)["features"]\n        return self.head(feats.mean(dim=2))       # average the time patches\n\n\nclass NameMatched:\n    """Reorders incoming electrodes into the trained order, zero-filling gaps.\n    Supports either a single model or a list of ensemble members whose L2-normalised\n    embeddings are averaged and re-normalised."""\n\n    def __init__(self, models, channels, in_names, device):\n        self.models = models if isinstance(models, list) else [models]\n        self.device = device\n        lookup = {n.casefold(): i for i, n in enumerate(in_names)}\n        self.src = [lookup.get(c.casefold(), -1) for c in channels]\n        missing = [c for c, i in zip(channels, self.src) if i < 0]\n        print(\n            f"[REVE-LoRA] matched {len(channels) - len(missing)}/{len(channels)} "\n            f"training channels by name among {len(in_names)} inputs; "\n            f"zero-filled: {missing if missing else \'none\'}; REVE uses "\n            f"{len(self.models[0].used)} with known positions"\n            + (f" ({len(self.models)} ensemble members)" if len(self.models) > 1 else "")\n        )\n\n    @torch.no_grad()\n    def predict(self, X):\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        if len(self.models) == 1:\n            self.models[0].eval()\n            return self.models[0](out)\n        preds = []\n        for m in self.models:\n            m.eval()\n            preds.append(F.normalize(m(out).float(), dim=-1))\n        return F.normalize(torch.stack(preds).mean(0), dim=-1)\n\n\nclass Solver(CompetSolver):\n\n    name = "REVE-LoRA-per-electrode"\n\n    requirements = ["pip::braindecode"]\n\n    def load_model(self, meta):\n        sub = meta["submission_dir"]\n        os.environ["REVE_POSITIONS_PATH"] = str(sub)   # bundled; no download at eval\n        in_names = meta.get("ch_names") or [c["ch_name"] for c in meta["chs_info"]]\n        if (sub / "ensemble_config.json").exists():\n            ecfg = json.loads((sub / "ensemble_config.json").read_text())\n            enc_state = torch.load(sub / "encoder.pt", map_location="cpu",\n                                   weights_only=True)\n            models = []\n            for mcfg in ecfg["members"]:\n                m = RevePerChannel(\n                    ecfg["channels"], ecfg["n_outputs"], meta["sfreq"],\n                    rank=mcfg["rank"], alpha=mcfg["alpha"],\n                    shrink=mcfg["shrink"], dropout=mcfg["dropout"],\n                    ffn_rank=mcfg.get("ffn_rank", 0), ffn_alpha=mcfg.get("ffn_alpha", 0),\n                    head_kind=mcfg.get("head_kind", "perelec"), mq_k=mcfg.get("mq_k", 2),\n                    encoder_state=enc_state,\n                )\n                tstate = torch.load(sub / mcfg["file"], map_location=meta["device"],\n                                    weights_only=True)\n                res = m.load_state_dict(tstate, strict=False)\n                absent = [k for k in tstate if k not in m.state_dict()]\n                if res.unexpected_keys or absent:\n                    raise RuntimeError(f"member {mcfg[\'file\']} mismatch: "\n                                       f"{(res.unexpected_keys or absent)[:4]}")\n                models.append(m.to(meta["device"]).eval())\n            return NameMatched(models, ecfg["channels"], in_names, meta["device"])\n\n        cfg = json.loads((sub / "reve_config.json").read_text())\n        model = RevePerChannel(\n            cfg["channels"], cfg["n_outputs"], meta["sfreq"], rank=cfg["rank"],\n            alpha=cfg["alpha"], shrink=cfg["shrink"], dropout=cfg["dropout"],\n            ffn_rank=cfg.get("ffn_rank", 0), ffn_alpha=cfg.get("ffn_alpha", 0),\n            head_kind=cfg.get("head_kind", "perelec"), mq_k=cfg.get("mq_k", 2),\n        )\n        state = torch.load(sub / "weights.pt", map_location=meta["device"],\n                           weights_only=True)\n        missing, unexpected = model.load_state_dict(state, strict=False)\n        if unexpected or missing:\n            raise RuntimeError(f"weights mismatch: missing {missing[:4]}, "\n                               f"unexpected {unexpected[:4]}")\n        model = model.to(meta["device"]).eval()\n        return NameMatched(model, cfg["channels"], in_names, meta["device"])\n'

stub = types.ModuleType('benchmark_utils.base_solver')
stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
sys.modules['benchmark_utils.base_solver'] = stub
ns = {}
exec(compile(SUBMISSION_PY, 'submission.py', 'exec'), ns)


In [ ]:
if RUN_TRAINING:
    from braindecode.models import REVE
    enc = REVE.from_pretrained('brain-bzh/reve-base', n_outputs=DIM, n_chans=len(CH),
                               n_times=200, sfreq=200.0, chs_info=[{'ch_name': c} for c in CH])
    enc_state = {k: v for k, v in enc.state_dict().items() if not k.startswith('final_layer')}
    del enc

    model = ns['RevePerChannel'](CH, DIM, IN_SFREQ, rank=LORA_RANK, alpha=LORA_ALPHA,
                                 ffn_rank=LORA_RANK_FFN, ffn_alpha=LORA_ALPHA_FFN,
                                 shrink=SHRINK, dropout=DROPOUT,
                                 head_kind=HEAD,
                                 encoder_state=enc_state).to(DEVICE)

    # Every pretrained tensor must now sit in the model, under its LoRA-wrapped name.
    msd = model.state_dict()
    def wrapped(k):
        # An adapter renames the module it sits on: .to_qkv -> .to_qkv.base, and
        # .net.1 -> .net.1.base for the feed-forward ones. Try each form; whichever
        # exists in the model (adapted or untouched) is where this key lives.
        k = 'net.' + k
        for t_ in ('to_qkv', 'to_out', 'net.1', 'net.3'):
            kk = k.replace(f'.{t_}.', f'.{t_}.base.')
            if kk in msd:
                return kk
        return k
    bad = [k for k, v in enc_state.items()
           if wrapped(k) not in msd or not torch.equal(msd[wrapped(k)].cpu(), v.cpu())]
    adapters = model.adapter_parameters()
    head_params = list(model.head.parameters())
    n_ad = sum(p.numel() for p in adapters)
    n_hd = sum(p.numel() for p in head_params)
    n_all = sum(p.numel() for p in model.parameters())
    print(f'pretrained tensors loaded: {len(enc_state) - len(bad)}/{len(enc_state)}'
          + (f'   MISMATCH: {bad[:4]}' if bad else ''))
    print(f'REVE uses {len(model.used)}/{len(CH)} electrodes with known positions; '
          f'LoRA on {model.n_lora} attention + {model.n_ffn} feed-forward projections '
          f'(expect 44 + 44 with both on)')
    print(f'trainable: adapters {n_ad / 1e6:.2f}M + head {n_hd / 1e6:.2f}M '
          f'of {n_all / 1e6:.1f}M total')
    assert not bad, 'pretrained encoder did not load; do not train'


## Step 3 — train

Three views per window, mean loss. Watch the gap between train and validation loss, and epoch 1's time (projected total printed).

In [ ]:
if RUN_TRAINING:
    EMB_T = torch.tensor(emb, device=DEVICE)


    def clip_loss(pred, idx):
        logits = (F.normalize(pred.float(), dim=-1)
                  @ F.normalize(EMB_T[idx], dim=-1).T) / TEMPERATURE
        same = idx[:, None] == idx[None, :]
        same.fill_diagonal_(False)
        logits = logits.masked_fill(same, -1e4)
        return F.cross_entropy(logits, torch.arange(len(pred), device=pred.device))


    def views(x):
        """The window, a noisy copy, a channel-shuffled copy (signals move between electrodes,
        positions stay put, time axis untouched). Originals only when AUGMENT is False."""
        out = [x]
        if AUGMENT:
            out.append(x + NOISE_FRAC * x.std(-1, keepdim=True) * torch.randn_like(x))
            order = torch.argsort(torch.rand(x.shape[0], x.shape[1], device=x.device), dim=1)
            out.append(x.gather(1, order[:, :, None].expand_as(x)))
        return out


    @torch.no_grad()
    def infer(X, bs=256):
        model.eval()
        out = []
        for i in range(0, len(X), bs):
            with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
                out.append(model(torch.from_numpy(X[i:i + bs]).to(DEVICE)).float())
        return torch.cat(out)


    fit_idx, val_idx = np.where(~IV & SEEN)[0], np.where(IV & SEEN)[0]
    Xv, yv = Wtr[0][val_idx], Wtr[1][val_idx]     # validation is read into RAM once (20%)
    yv_t = torch.tensor(yv.astype('int64'), device=DEVICE)
    opt = torch.optim.AdamW([{'params': adapters, 'lr': LR_ADAPTER},
                             {'params': head_params, 'lr': LR_HEAD}], weight_decay=0.0)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    trainable = {n for n, p in model.named_parameters() if n.endswith(('.A', '.B'))
                 or n.startswith('head.')}
    print(f'chance loss {np.log(BATCH):.4f}')

    best, best_state, hist = -1, None, []
    bad, loss_up, vl_best, stopped_at = 0, 0, 1e9, None
    for ep in range(1, EPOCHS + 1):
        on = ep > HEAD_ONLY_EPOCHS
        for p in adapters:
            p.requires_grad_(on)
        model.train()
        tot, t0 = 0.0, time.time()
        perm = rng.permutation(len(fit_idx))
        batches = [fit_idx[perm[i:i + BATCH]] for i in range(0, len(perm) - 7, BATCH)]
        for bi, rows in enumerate(batches, 1):
            if bi in (20, 100) or bi % 500 == 0:
                print(f'    batch {bi}/{len(batches)}  {time.time() - t0:.0f}s  projected epoch '
                      f'{(time.time() - t0) / bi * len(batches) / 60:.1f} min; projected {EPOCHS} epochs '
                      f'{(time.time() - t0) / bi * len(batches) * EPOCHS / 60:.0f} min')
            b = np.sort(rows)                      # sorted rows read faster from disk
            xb = torch.from_numpy(Wtr[0][b]).to(DEVICE).float()
            yb = torch.from_numpy(Wtr[1][b].astype('int64')).to(DEVICE)
            loss = 0.0
            for v in views(xb):
                with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
                    pred = model(v)
                loss = loss + clip_loss(pred, yb)
            loss = loss / (3 if AUGMENT else 1)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += float(loss) * len(b)
        pv = infer(Xv)
        vl = float(np.mean([float(clip_loss(pv[i:i + BATCH], yv_t[i:i + BATCH]))
                            for i in range(0, len(pv) - 7, BATCH)]))
        r = rank_pct(pv.cpu().numpy(), yv, emb)
        hist.append((ep, tot / len(fit_idx), vl, r))
        last_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                      if k in trainable}
        improved = r > best + MIN_DELTA
        if r > best:
            best = r
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                          if k in trainable}
            torch.save(best_state, WEIGHTS / f'{RUN_TAG}_best_so_far.pt')   # survives a Colab disconnect
        if ep == 1:
            print(f'  epoch 1 took {time.time() - t0:.0f}s; projected total for {EPOCHS} epochs '
                  f'about {(time.time() - t0) * EPOCHS / 60:.0f} min')
        print(f'  epoch {ep:2d}{"" if on else " (head only)"}  train loss {tot / len(fit_idx):.4f}  '
              f'val loss {vl:.4f}  val rank {r:.4f}{"  *" if r == best else ""}  '
              f'{time.time() - t0:.0f}s')
        vl_best = min(vl_best, vl)
        bad = 0 if improved else bad + 1
        loss_up = loss_up + 1 if vl > vl_best + DIVERGE_LOSS else 0
        if on and (bad >= PATIENCE or loss_up >= 2):
            stopped_at = ep
            why = (f'no validation-rank gain for {bad} epochs' if bad >= PATIENCE else
                   f'val loss {vl:.4f} is above its minimum {vl_best:.4f} by more than {DIVERGE_LOSS}')
            print(f'  early stop at epoch {ep}: {why}; best was epoch {max(hist, key=lambda h: h[3])[0]}')
            break
    # Only the trained tensors (adapters, head) are restored; the frozen encoder is
    # already in place. Assigned so Jupyter does not print the long list of
    # encoder keys that were deliberately not part of best_state.
    _ = model.load_state_dict(best_state if KEEP == 'best' else last_state, strict=False)


## Step 4 — score it on held-out people

In [ ]:
if RUN_TRAINING:
    best_ep = max(hist, key=lambda h: h[3])
    vl_min = min(hist, key=lambda h: h[2])
    print(f'best val rank {best_ep[3]:.4f} at epoch {best_ep[0]} (KEEP = {KEEP!r}); '
          f'lowest val loss {vl_min[2]:.4f} at epoch {vl_min[0]}')

    pred = infer(Wte[0]).cpu().numpy()
    r_te = rank_pct(pred, Wte[1], emb)
    a_te, ch = top5(pred, Wte[1], emb)
    agg = {}
    for p_, y_, s_ in zip(pred, Wte[1], Wte[2]):
        agg.setdefault((s_, y_), []).append(p_)
    ks = sorted(agg)
    a_agg, _ = top5(np.stack([np.mean(agg[k_], 0) for k_ in ks]), np.array([k_[1] for k_ in ks]), emb)
    e = None
    print(f'\n  {"seen people, new images":<24} rank {r_te:.4f}   top-5 {a_te:5.2f}% (per-window)'
          f'   {a_agg:5.2f}% (subject-agg)   chance {ch:.2f}%')
    if HOLDOUT_PEOPLE:
        hm = np.isin(Wte[2], [str(p_) for p_ in HOLDOUT_PEOPLE])
        r_h = rank_pct(pred[hm], Wte[1][hm], emb)
        a_h, ch_h = top5(pred[hm], Wte[1][hm], emb)
        agg_h = {}
        for p_, y_, s_ in zip(pred[hm], Wte[1][hm], Wte[2][hm]):
            agg_h.setdefault((s_, y_), []).append(p_)
        ks_h = sorted(agg_h)
        a_hagg, _ = top5(np.stack([np.mean(agg_h[k_], 0) for k_ in ks_h]),
                         np.array([k_[1] for k_ in ks_h]), emb)
        print(f'\n  HELD-OUT {str(HOLDOUT_PEOPLE):<9} rank {r_h:.4f}   '
              f'top-5 {a_h:5.2f}%   {a_hagg:5.2f}% (subject-agg)   chance {ch_h:.2f}%   '
              f'{hm.sum():,} windows, {len(ks_h)} images - never fitted or validated')

    nullv = rank_null(infer(Xv).cpu().numpy(), yv)
    thr = float(np.percentile(nullv, 95))
    climb = best_ep[3] - hist[0][3]
    TRENDING = bool(best_ep[3] > thr and best_ep[0] > 1 and climb > 2 * nullv.std())
    print(f'\nval rank under permuted labels: mean {nullv.mean():.4f}  sd {nullv.std():.4f}  '
          f'95th {thr:.4f}; best {best_ep[3]:.4f} '
          f'({(best_ep[3] - nullv.mean()) / max(nullv.std(), 1e-9):.1f} sd above)')
    if TRENDING and e:
        verdict = 'beats' if a_te > e['top5'] else 'does not beat'
        print(f'Trending, and REVE {verdict} EEGNet on the same data '
              f'({a_te:.2f}% vs {e["top5"]:.2f}%).')
    elif TRENDING:
        print('Trending: it learns something that generalises to unseen images.')
    else:
        print('Not trending: validation rank never clears its own null.')

    torch.save(best_state if KEEP == 'best' else last_state, WEIGHTS / f'{RUN_TAG}_trainable.pt')
    print(f'saved -> {WEIGHTS / f"{RUN_TAG}_trainable.pt"} (adapters and head only)')


## Step 5 — package

Writes the weights (REVE plus adapters plus head), the config and the submission file, reloads them the way the harness does, and zips only if the checks pass.

In [ ]:
def package(model, tag, test_top5, rank, alpha, shrink, dropout,
            ffn_rank=0, ffn_alpha=0, head_kind=None):
    """Write weights, config and solver; reload them the way the harness does; zip
    only if the reloaded submission reproduces the model it came from."""
    head_kind = HEAD if head_kind is None else head_kind
    import importlib.util
    out = BASE / 'submissions' / f'track1_{tag}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    torch.save({k: v.cpu() for k, v in model.state_dict().items()}, out / 'weights.pt')
    (out / 'reve_config.json').write_text(json.dumps(dict(
        channels=CH, n_outputs=DIM, rank=rank, alpha=alpha, shrink=shrink,
        dropout=dropout, ffn_rank=ffn_rank, ffn_alpha=ffn_alpha,
        head_kind=head_kind,
        trained_on=f'Alljoined-1.6M ({tag})', test_top5=test_top5), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY)
    bank = [p for p in (Path.home() / 'mne_data').glob('*reve*positions*.json')]
    if not bank:
        import huggingface_hub
        bank = [Path(huggingface_hub.hf_hub_download('brain-bzh/reve-positions',
                                                     'positions.json'))]
    shutil.copy(bank[0], out / 'reve_positions.json')

    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    def run(names, X):
        m = mod.Solver().load_model({'ch_names': list(names), 'chs_info': None,
                                     'sfreq': IN_SFREQ, 'submission_dir': out,
                                     'device': DEVICE, 'n_outputs': DIM})
        return m.predict(torch.from_numpy(X.astype('float32'))).cpu().numpy()

    Xs = Wte[0][:8].astype('float32')
    base = run(CH, Xs)
    # Reference in full precision: the submission runs without autocast, so a
    # half-precision reference would only measure fp16 rounding.
    with torch.no_grad():
        model.eval()
        ref = model(torch.from_numpy(Xs).to(DEVICE)).float().cpu().numpy()
    perm = rng.permutation(len(CH))
    shuffled = run([CH[i] for i in perm], Xs[:, perm])
    wide = run(list(CH) + [f'X{i}' for i in range(31)],
               np.concatenate([Xs, rng.standard_normal((8, 31, Xs.shape[-1])).astype('float32')], 1))
    drop = list(CH[2:])      # two electrodes missing
    part = run(drop, Xs[:, [CH.index(c) for c in drop]])
    checks = [
        ('same as the model it came from', np.allclose(base, ref, atol=1e-3),
         f'max |diff| {np.abs(base - ref).max():.1e}'),
        ('order-independent', np.allclose(base, shuffled, atol=1e-4),
         f'max |diff| {np.abs(base - shuffled).max():.1e}'),
        ('extra electrodes', np.allclose(base, wide, atol=1e-4), 'ignored, output identical'),
        ('missing electrodes', part.shape == base.shape and np.isfinite(part).all(),
         'zero-filled, still runs'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if not all(c[1] for c in checks):
        print('\nNot zipping: a check failed.')
        return None
    zpath = shutil.make_archive(str(out), 'zip', out)
    print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e6:.0f} MB)')
    return zpath


if RUN_TRAINING and MAKE_ZIP:
    if TRENDING or globals().get('FORCE_ZIP'):
        package(model, RUN_TAG, a_te, LORA_RANK, LORA_ALPHA, SHRINK, DROPOUT,
               LORA_RANK_FFN, LORA_ALPHA_FFN)
    else:
        print('Not packaging: the run did not clear its null. Set FORCE_ZIP = True to override.')
